In [ ]:
# Accept parameters passed from orchestration notebook via dbutils.notebook.run()
# These simulate DAB variables in the bundle deployment

try:
    # Get parameters from dbutils.widgets (passed by dbutils.notebook.run)
    catalog_name = dbutils.widgets.get("catalog_name")
    schema_prefix = dbutils.widgets.get("schema_prefix")
    print(f"Using parameters from orchestration:")
    print(f"  catalog_name: {catalog_name}")
    print(f"  schema_prefix: {schema_prefix}")
except Exception:
    # Fallback to default values if not called from orchestration
    catalog_name = "dev_catalog"
    schema_prefix = "slv_cdm_hrs"
    print(f"Using default values (not called from orchestration):")
    print(f"  catalog_name: {catalog_name}")
    print(f"  schema_prefix: {schema_prefix}")

This notebook is used to load and test the HRS RESPONDENT table.  It extracts distinct HHIDPN values from the RAND longitudinal data and populate the table.

**Purpose:** Load the HRS RESONDENT reference table.

**Source Table:** `dev_catalog.brz_raw_hrs.randhrs1992_2022v1`  
**Target Table:** `dev_catalog.slv_cdm_hrs.resondent`
**Load Script:** `../../sql/dml/load_hrs_respondent_data.sql`
**Validation Script:** `../../sql/validataion/verify_hrs_respondent_data.sql`

**Process:**
1. Clear/truncate the HRS RESONDENT table .
2. Extract distinct HHIDPN values from SOURCE data and load the TARGET table.
3. Validate the table data.
4. Display summary stats.

In [ ]:
# -----------------------------------------------------------------------------
# Initialize Notebook Configuration
# -----------------------------------------------------------------------------
# For Asset Bundles
#   Instead of relying on relative paths, add the bundle root to Python's path.
#   In each notebook that imports src, add this before the import:
import sys
sys.path.append("/Workspace/Users/peteperez.lv@gmail.com/.bundle/hrs_dbx_repo/default/files")

dbutils.widgets.dropdown(
    "truncate_table",
    "true",
    ["true", "false"]
)

TRUNCATE_TABLE = dbutils.widgets.get("truncate_table").lower() == "true"

TARGET_TABLE = f"{catalog_name}.{schema_prefix}.hub_respondent"

LOAD_SQL = "../../sql/dml/load_hrs_respondent_data.sql"

VALIDATION_SQL = "../../sql/validation/validate_hrs_respondent_data.sql"

SOURCE_TABLE = "dev_catalog.brz_raw_hrs.randhrs1992_2022v1"

In [ ]:
# Step 1:
# Clear existing respondent data if needed (use with caution)
# Uncomment the line below to truncate the table before loading 

if TRUNCATE_TABLE:
    print("======================================================")
    print("Step 1 - TRUNCATE")
    print("======================================================")

    try:
        spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
        print("✓ Completed")
    except Exception as e:
        print(f"❌ TRUNCATE failed: {e}")
        raise

else:
    print("Table not found.  Skipping table truncation.")

In [ ]:
# Step 2
# Load distinct data to the TARGET_TABLE

from pathlib import Path
import re

print("======================================================")
print("Step 2 - LOAD DATA")
print("======================================================")
try:
    # Read SQL file
    sql_path = Path(LOAD_SQL)
    sql_text = sql_path.read_text()
    
    # Replace all IDENTIFIER(CONCAT(...)) patterns with the actual target table name
    # This avoids the RDD_BASED error with Spark Connect
    # It's not that the table is RDD-based - it's that dynamic table name resolution 
    # #with parameters creates an RDD-based query plan. Spark Connect requires all 
    # table references to be concrete at analysis time.
    sql_text = re.sub(
        r"IDENTIFIER\s*\(\s*CONCAT\s*\([^)]+\)\s*\)",
        TARGET_TABLE,
        sql_text,
        flags=re.IGNORECASE
    )
    
    # Split and execute statements
    statements = [stmt.strip() for stmt in sql_text.split(';') if stmt.strip()]
    
    for i, stmt in enumerate(statements, 1):
        print(f"  Executing statement {i}/{len(statements)}")
        spark.sql(stmt)
    
    print("✓ Completed")
except Exception as e:
    print(f"❌ Load failed: {e}")
    raise


In [ ]:
# Step 3: Verify the TARGET_TABLE.
from pathlib import Path

print("======================================================")
print("Step 3 - Validation")
print("======================================================")
try:
    # Read SQL file
    sql_path = Path(VALIDATION_SQL)
    sql_text = sql_path.read_text()
    
    # Split and execute statements with parameter binding
    statements = [stmt.strip() for stmt in sql_text.split(';') if stmt.strip()]
    
    for i, stmt in enumerate(statements, 1):
        print(f"  Executing statement {i}/{len(statements)}")
        result = spark.sql(stmt, args={"catalog_name": catalog_name, "schema_prefix": schema_prefix})
        display(result)
    
    print("✓ Completed")
except Exception as e:
    print(f"❌ Validation failed: {e}")
    raise

In [ ]:
# Step 4 - Display summary statistics

source_count = spark.sql("""
    SELECT COUNT(DISTINCT HHIDPN) as distinct_hhidpn
    FROM dev_catalog.brz_raw_hrs.randhrs1992_2022v1
    WHERE HHIDPN IS NOT NULL
""").collect()[0][0]

target_count = spark.sql(f"""
    SELECT COUNT(*) as respondent_count
    FROM {TARGET_TABLE}
""").collect()[0][0]

print("=" * 60)
print("HRS HHIDPN DATA LOAD SUMMARY")
print("=" * 60)
print(f"Distinct HHIDPN values in source: {source_count}")
print(f"Total records in HRS Respondent table:      {target_count}")
print("=" * 60)

if source_count == target_count:
    print("✓ SUCCESS: All distinct HRS HHIDPN loaded")
else:
    print(f"⚠ WARNING: Mismatch detected. Please review.")

: 